In [1]:
import os
import yaml
import torch
from torch.utils.data import DataLoader
import pandas as pd
from data.regression.data import TestCellDataset, test_transform_2
from utils import init_env, sliding_window_prediction_save, evaluate_predictions
from models import build_model

def load_config(config_path):
    """Load configuration from a YAML file."""
    with open(config_path, 'r') as file:
        return yaml.load(file, Loader=yaml.FullLoader)

# Load configuration directly
config_path = './config.yml'  # Update with the actual path
config = load_config(config_path)

In [2]:
# Initialize environment
init_env()
model_name = 'densenet121'
exp_name = f"exp_{model_name}_only_real_data_cutmix"
     
    

In [3]:
# Test Dataset and DataLoader
# test_path = config['test_txt_file']  # Path to test.txt from config
test_path = "splits/real_data/test.txt"  # Path to test.txt from config

test_dataset = TestCellDataset(txt_file=test_path, transform=test_transform_2)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4)

print(f"Loaded test dataset with {len(test_dataset)} samples.")

Loaded test dataset with 108 samples.


In [4]:
!ls checkpoints

densenet121_fold_1_best_model.pth      efficientnet_b0_fold_5_best_model.pth
densenet121_fold_2_best_model.pth      efficientnet_v2_b0_fold_1_best_model.pth
densenet121_fold_3_best_model.pth      resnet50_fold_0_best_model.pth
densenet121_fold_4_best_model.pth      resnet50_fold_1_best_model.pth
densenet121_fold_5_best_model.pth      resnet50_fold_2_best_model.pth
efficientnet_b0_fold_0_best_model.pth  resnet50_fold_3_best_model.pth
efficientnet_b0_fold_1_best_model.pth  resnet50_fold_4_best_model.pth
efficientnet_b0_fold_2_best_model.pth  resnet50_fold_5_best_model.pth
efficientnet_b0_fold_3_best_model.pth  swin_t_fold_1_best_model.pth
efficientnet_b0_fold_4_best_model.pth


In [5]:
# # Load Pre-Trained Models
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model_dir = 'checkpoints'  # Directory containing pre-trained models

# models = []
# for fold in range(1, 6):  # 5 Folds
#     model_path = os.path.join(model_dir, f'resnet50_fold_{fold}_best_model.pth')
#     # model = build_model(config['model_name'])
#     model = build_model('resnet50')
#     model.load_state_dict(torch.load(model_path))
#     model.to(device)
#     model.eval()
#     models.append(model)

# print(f"Loaded {len(models)} pre-trained models.")

import os
import torch

# Load Pre-Trained Models
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
experiment_dir = f'experiments/{exp_name}'  # Directory containing experiment folders

models = []
for fold_dir in os.listdir(experiment_dir):  # Iterate over all fold directories
    fold_path = os.path.join(experiment_dir, fold_dir)
    if os.path.isdir(fold_path):  # Ensure it's a directory
        # Locate the best model file dynamically
        best_model_file = None
        for file_name in os.listdir(fold_path):
            if file_name.startswith("resnet50_ACP5_") and file_name.endswith(".pth"):  # Match file pattern
                best_model_file = os.path.join(fold_path, file_name)
                break  # Stop at the first match (assuming only one best model file per fold)

        if best_model_file:
            print(f"Loading model from {best_model_file}")
            # Build and load the model
            model = build_model(model_name)  # Assume build_model is already imported
            model.load_state_dict(torch.load(best_model_file, map_location=device))
            model.to(device)
            model.eval()
            models.append(model)
        else:
            print(f"No best model found in {fold_path}")

print(f"Loaded {len(models)} pre-trained models.")

Loading model from experiments/exp_densenet121_only_real_data_cutmix/fold_3/resnet50_ACP5_22.0_MAE_17.18.pth
Loading model from experiments/exp_densenet121_only_real_data_cutmix/fold_4/resnet50_ACP5_52.0_MAE_9.02.pth
Loading model from experiments/exp_densenet121_only_real_data_cutmix/fold_1/resnet50_ACP5_48.0_MAE_11.9.pth
Loading model from experiments/exp_densenet121_only_real_data_cutmix/fold_5/resnet50_ACP5_38.0_MAE_16.12.pth
Loading model from experiments/exp_densenet121_only_real_data_cutmix/fold_2/resnet50_ACP5_24.0_MAE_17.34.pth
No best model found in experiments/exp_densenet121_only_real_data_cutmix/analysis
Loaded 5 pre-trained models.


In [6]:
# Prediction Parameters
window_size = (200, 200)
stride = 200
exp_name = f"{exp_name}_test_prediction"
os.makedirs(exp_name, exist_ok=True)

# Predict with each model
all_predictions = []
for fold, model in enumerate(models, start=1):
    print(f"Predicting with model fold {fold}...")
    predictions, _ = sliding_window_prediction_save(
        model=model,
        dataloader=test_loader,
        device=device,
        window_size=window_size,
        stride=stride,
        exp_name=f"{exp_name}_fold_{fold}"
    )
    all_predictions.append(predictions)

print("Predictions completed.")

Predicting with model fold 1...
Processed 220815_GFP-AHPC_A_Ki67_F8_DAPI_ND1_20x.tiff (1/108)
Processed 220909_GFP-AHPC_D_Ki67_F8_DAPI_ND1_20x.tiff (1/108)
Processed 220912_GFP-AHPC_C_Map2AB_F9_DAPI_ND1_20x.tiff (1/108)
Processed 220815_GFP-AHPC_C_Ki67_F1_DAPI_ND1_20x.tiff (1/108)
Processed 220815_GFP-AHPC_A_Nestin_F5_DAPI_ND1_20x.tiff (1/108)
Processed 220815_GFP-AHPC_D_Ki67_F5_DAPI_ND1_20x.tiff (1/108)
Processed 220909_GFP-AHPC_D_GFAP_F6_DAPI_ND1_20x.tiff (1/108)
Processed 220816_GFP-AHPC_D_RIP_F1_DAPI_ND1_20x.tiff (1/108)
Processed 220815_GFP-AHPC_A_TuJ1_F2_DAPI_ND1_20x.tiff (1/108)
Processed 220815_GFP-AHPC_A_TuJ1_F9_DAPI_ND1_20x.tiff (1/108)
Processed 220909_GFP-AHPC_D_MAP2ab_F9_DAPI_ND1_20x.tiff (1/108)
Processed 220909_GFP-AHPC_D_Nestin_F9_DAPI_ND1_20x.tiff (1/108)
Processed 220816_GFP-AHPC_B_RIP_F7_DAPI_ND1_20x.tiff (1/108)
Processed 220815_GFP-AHPC_C_MAP2ab_F10_DAPI_ND1_20x.tiff (1/108)
Processed 220909_GFP-AHPC_B_GFAP_F6_DAPI_ND1_20x.tiff (1/108)
Processed 220815_GFP-AHPC_D_T

In [7]:
!ls test_images_fall2024_unzipped

__MACOSX  images


In [8]:
# # Combine and Save Predictions
image_paths = [os.path.basename(path) for path, _ in test_dataset]
combined_predictions = {f"Fold_{i+1}_Prediction": preds for i, preds in enumerate(all_predictions)}
combined_predictions['Image_Name'] = image_paths

results_df = pd.DataFrame(combined_predictions)
results_csv = os.path.join(exp_name, "test_results.csv")
results_df.to_csv(results_csv, index=False)

print(f"Predictions saved to {results_csv}")

Predictions saved to exp_densenet121_only_real_data_cutmix_test_prediction/test_results.csv


In [9]:
# # Evaluate Predictions (if ground truth is available)
# if config.get("csv_directory"):
#     test_mae_5, test_acp_5 = evaluate_predictions(results_csv, threshold=0.05)
#     print(f"Test MAE@5%: {test_mae_5}, Test ACP@5%: {test_acp_5}")
#     print("*" * 10)
#     test_mae_10, test_acp_10 = evaluate_predictions(results_csv, threshold=0.1)
#     print(f"Test MAE@10%: {test_mae_10}, Test ACP@10%: {test_acp_10}")

In [10]:
import pandas as pd

# Load the test predictions CSV file
test_predictions_file = f"{exp_name}/test_results.csv"  
df = pd.read_csv(test_predictions_file)

# Ensure the structure matches expectations
if "Image_Name" not in df.columns or len(df.columns) < 6:
    raise ValueError("The input CSV file must contain five fold prediction columns and an 'Image_Name' column.")

# Extract fold predictions and image names
fold_columns = [col for col in df.columns if col.startswith("Fold_")]
df["Prediction"] = df[fold_columns].mean(axis=1)  # Average predictions across folds

# Create the final submission dataframe
submission_df = df[["Image_Name", "Prediction"]].rename(columns={"Image_Name": "filename"})

# Save the final submission file
submission_file = f"{exp_name}/submission.csv"
submission_df.to_csv(submission_file, index=False)
print(f"Submission file saved to: {submission_file}")

Submission file saved to: exp_densenet121_only_real_data_cutmix_test_prediction/submission.csv


In [11]:
# import os
# import pandas as pd
# import matplotlib.pyplot as plt

# # Dummy implementation for evaluate_predictions (replace with your actual function)
# def evaluate_predictions(results_csv, threshold=0.05):
#     # Load the CSV file
#     df = pd.read_csv(results_csv)
#     # Replace these calculations with your actual evaluation logic
#     mae = abs(df["Predictions"] - df["Targets"]).mean()  # Example MAE
#     acp = ((abs(df["Predictions"] - df["Targets"]) <= threshold * df["Targets"]).mean()) * 100  # Example ACP
#     return mae, acp

# # Path to the parent directory
# base_dir = "experiments/exp_resnet50_synthetic_and_real_data"

# # Initialize lists to store results
# mae_5_list = []
# acp_5_list = []
# mae_10_list = []
# acp_10_list = []

# # Iterate through the folds
# for fold in range(1, 6):
#     results_csv = os.path.join(base_dir, f"fold_{fold}", "test_results.csv")
#     if not os.path.exists(results_csv):
#         print(f"File not found: {results_csv}")
#         continue

#     print(f"Evaluating {results_csv}...")
#     # Evaluate for threshold 0.05
#     test_mae_5, test_acp_5 = evaluate_predictions(results_csv, threshold=0.05)
#     mae_5_list.append(test_mae_5)
#     acp_5_list.append(test_acp_5)

#     # Evaluate for threshold 0.1
#     test_mae_10, test_acp_10 = evaluate_predictions(results_csv, threshold=0.1)
#     mae_10_list.append(test_mae_10)
#     acp_10_list.append(test_acp_10)

# # Calculate mean and standard deviation
# results_summary = {
#     "Metric": ["MAE@5%", "ACP@5%", "MAE@10%", "ACP@10%"],
#     "Mean": [sum(mae_5_list) / len(mae_5_list), sum(acp_5_list) / len(acp_5_list),
#              sum(mae_10_list) / len(mae_10_list), sum(acp_10_list) / len(acp_10_list)],
#     "Std": [pd.Series(mae_5_list).std(), pd.Series(acp_5_list).std(),
#             pd.Series(mae_10_list).std(), pd.Series(acp_10_list).std()],
# }

# # Display results
# results_df = pd.DataFrame(results_summary)
# print(results_df)

# # Plot box plots for ACP@5% and ACP@10%
# plt.figure(figsize=(8, 6))
# plt.boxplot([acp_5_list, acp_10_list], labels=["ACP@5%", "ACP@10%"], boxprops=dict(linewidth=2), medianprops=dict(linewidth=2, color="red"))
# plt.title("Box Plot of ACP@5% and ACP@10%", fontsize=16, fontweight="bold")
# plt.ylabel("Accuracy Percentage (%)", fontsize=14)
# plt.xticks(fontsize=12)
# plt.yticks(fontsize=12)
# plt.grid(axis="y", linestyle="--", alpha=0.7)
# plt.tight_layout()
# plt.savefig("acp_box_plot.png", dpi=300)
# plt.show()